# WaterOnt: all LLM configurations in one session

Runs, in one session, every LLM-assisted configuration of the paper on the 24 questions: flat-text RAG with TF-IDF and with dense-vector retrieval, the full-context LLM (every pipe description at once), and WaterOnt GraphRAG, plus the retrieval analysis (TF-IDF, BM25, dense). The pipe descriptions now state the supply zone and the monitoring status.

**Before you start**
1. Runtime → **Disconnect and delete runtime**.
2. Your `ANTHROPIC_API_KEY` secret is already set (notebook access on).
3. Have the latest `WaterOnt-full-repo.zip` on your computer; step 2 asks for it.
4. Run the cells in order. Step 1 may restart the runtime once; if so, continue from step 2. About 125 API calls in total, roughly 45 minutes.

In [ ]:
# 1. Install packages (keeps NumPy compatible with Colab's SciPy)
import os, subprocess, sys
os.chdir("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2.0,<2.3",
                "wntr", "rdflib", "anthropic", "sentence-transformers"], check=True)
on_disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
import numpy
if numpy.__version__ != on_disk:
    print(f"NumPy changed ({numpy.__version__} -> {on_disk}): restarting. Continue from step 2.")
    os.kill(os.getpid(), 9)
print("Step 1 done. NumPy", numpy.__version__)

In [ ]:
# 2. Get the code from WaterOnt-full-repo.zip
import os, sys, shutil, zipfile
from google.colab import files
os.chdir("/content")
uploaded = files.upload()                       # choose WaterOnt-full-repo.zip
shutil.rmtree("/content/WaterOnt", ignore_errors=True)
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")
os.chdir("/content/WaterOnt")
if "/content/WaterOnt" not in sys.path:
    sys.path.insert(0, "/content/WaterOnt")
for name in [m for m in sys.modules if m.startswith("wateront")]:
    del sys.modules[name]
import wateront.baselines as B
from wateront.config import Config
if not hasattr(B, "zone_text") or not hasattr(B.FlatRAGBaseline, "FULL_CONTEXT_NOTE"):
    raise RuntimeError("This is an OLD WaterOnt-full-repo.zip. Download the latest one and rerun step 2.")
print("Step 2 done. Latest code; LLM model:", Config().llm_model)

In [ ]:
# 3. API key and a one-line test call. Must print "API replied: OK".
import os, anthropic
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip().strip("\"'")
reply = anthropic.Anthropic().messages.create(model=Config().llm_model, max_tokens=10,
                                              messages=[{"role": "user", "content": "Reply with OK"}])
print("Step 3 done. API replied:", reply.content[0].text)

In [ ]:
# 4. Smoke test: a supply-zone question, now answerable from the pipe descriptions
from wateront.pipeline import build_knowledge_graph
cfg = Config(seed=42)
res = build_knowledge_graph("data/networks/L-TOWN.inp", cfg, verbose=False)
flat = B.FlatRAGBaseline(res.kg, cfg, use_llm=True)
example = [d["text"] for d in flat.index.docs if d["kind"] == "asset" and "PRV-3" in d["text"]][0]
print("A pipe description now ends with:", example[-110:])
answer = flat.ask("Which pipes downstream of PRV-3 need intervention?")
print("TF-IDF RAG recommendations:", len(answer.recommendations), "| unsupported citations:", answer.unsupported_citations)

In [ ]:
# 5. Full run, all in this session (about 45 minutes)
import subprocess, sys
for script in ["paper/experiments_retrieval.py", "paper/experiments_full_context.py"]:
    print(f"=== {script}")
    run = subprocess.run([sys.executable, script], capture_output=True, text=True, cwd="/content/WaterOnt")
    print(run.stdout[-2500:])
    if run.returncode != 0:
        print(run.stderr[-3000:])
        raise RuntimeError(f"{script} failed; send the error above")
print("Step 5 done.")

In [ ]:
# 6. Results
import json, pandas as pd
print(json.load(open("paper/results/retrieval_meta.json")))
print(json.load(open("paper/results/full_context_meta.json")))
display(pd.concat([pd.read_csv("paper/results/retrieval_rag_summary.csv"),
                   pd.read_csv("paper/results/full_context_summary.csv")]).set_index("system").round(3))

In [ ]:
# 7. Download all new result files
import shutil
from google.colab import files
os.makedirs("/content/all_llm_results", exist_ok=True)
for f in ["retrieval_analysis.csv", "retrieval_rag_per_query.csv", "retrieval_rag_summary.csv", "retrieval_meta.json",
          "full_context_per_query.csv", "full_context_summary.csv", "full_context_meta.json"]:
    shutil.copy(f"/content/WaterOnt/paper/results/{f}", "/content/all_llm_results/")
shutil.make_archive("/content/all_llm_results", "zip", "/content/all_llm_results")
files.download("/content/all_llm_results.zip")